In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os
data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.hist( df['Delivery_Time'].dropna(),bins=30, color='coral',edgecolor='black')
plt.title('delivery distribution')
plt.xlabel('Delivery Time')
plt.ylabel('frequency')
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_clean= df.drop(columns=['Order_ID'])


In [ ]:
missing_percentage = (df.isnull().sum() )
print(missing_percentage)
df_clean = df.dropna(subset=['Weather', 'Traffic_Level ', 'Time_of_Day ','Courier_Experience_yrs'])
for col in ['Time_of_Day ', 'Courier_Experience_yrs']:
    df_clean[col] = df_clean[col].fillna('unknown')


In [ ]:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
import pandas as pd
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))




In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

In [ ]:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
lr_mae = []
f = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(f.split(X)):
  print(f"\nFold {fold_idx + 1}/{5}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

    # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)

print("Model trained!")
y_pred = model.predict(X_test)

print(f"MAE:  ${mae:,.2f}")

average_losses = np.mean(lr_mae, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MAE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist( y_test,bins=30, color='coral',edgecolor='black')
plt.title('delivery distribution')
plt.xlabel('Delivery Time')
plt.ylabel('frequency')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Task Bonus: Write your code here: